# Phase 1 master data

The master combines the six stocks and S&P 500 from Yahoo with VIX from Cboe. Returns use Close. Adjusted Close stays in the file for reference, and dividends are omitted. The source comparison is in [the Phase 1 analysis](../docs/phase_planning/phase_1_analysis.md); the adopted choices are in [methodology](../docs/methodology.md#phase-1-master-data).

The initial notebook inspection found 2,011 historical trading dates, 27 Cboe-only dates, and 14 stock returns with an absolute value above 15%. The reusable implementation below repeats those checks and saves the master and audit files. No models, thresholds, or preprocessing parameters are fitted here.

In [1]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display
from market_analytics.data.master import build_master, master_columns, sha256

ROOT = Path.cwd().resolve()
if not (ROOT / 'pyproject.toml').is_file():
    ROOT = ROOT.parent
assert (ROOT / 'pyproject.toml').is_file()
CONFIG = ROOT / 'configs/master_data.json'
settings = json.loads(CONFIG.read_text(encoding='utf-8'))
display(pd.DataFrame(settings['sources'].items(), columns=['Source', 'Pinned snapshot']))

,Source,Pinned snapshot
0,yahoo,data/raw/yahoo_finance/20261017T112037944190Z
1,cboe,data/raw/cboe/20261024T163804809102Z


## Historical validation and alignment

The build verifies the selected inputs against their acquisition manifests before calculating anything. Stock dates must match the S&P 500 exactly. Missing dates, duplicate dates, invalid prices, or invalid stock volume stop the build. Prices are never filled or interpolated.

This first call does not open Yahoo's unseen files. The shared Cboe history is hashed in full, then restricted to 2018-2025 before its prices are checked or aligned. The interim checkpoint keeps source fields, including dividends and splits.

In [2]:
historical_run = build_master(CONFIG, historical_only=True)
historical = pd.read_csv(ROOT / 'data/processed/master_historical.csv', index_col='Date', parse_dates=True, float_precision='round_trip')
assert historical.shape == (2011, 48)
assert list(historical.columns) == master_columns()
historical_hash = sha256(ROOT / 'data/processed/master_historical.csv')
display(historical.head(3))

,MSFT_Open,MSFT_High,MSFT_Low,MSFT_Close,MSFT_AdjClose,MSFT_Volume,MSFT_Return,GOOGL_Open,GOOGL_High,GOOGL_Low,...,META_Close,META_AdjClose,META_Volume,META_Return,GSPC_Close,GSPC_Return,VIX_Close,Year,Quarter,Month
Date,,,,,,,,,,,,,,,,,,,,,
2018-01-02,86.129997,86.309998,85.500000,85.949997,78.552040,22483800,NaN,52.651001,53.799000,52.651001,...,181.419998,179.698914,18151900,NaN,2695.810059,NaN,9.77,2018,1,1
2018-01-03,86.059998,86.510002,85.970001,86.349998,78.917618,26061400,0.465389,53.696499,54.805000,53.671501,...,184.669998,182.918076,16886600,1.791423,2713.060059,0.639882,9.15,2018,1,1
2018-01-04,86.589996,87.660004,86.570000,87.110001,79.612198,21912000,0.880141,54.854500,55.203999,54.713001,...,184.330002,182.581284,13880900,-0.184110,2723.989990,0.402864,9.22,2018,1,1


In [3]:
excluded = pd.read_csv(ROOT / 'data/interim/excluded_rows.csv')
assert len(excluded) == 27
assert excluded['Period'].eq('historical').all()
missing = historical.isna().sum()
assert int(missing.sum()) == 7
assert historical.iloc[1:].notna().all().all()
print('Excluded Cboe-only historical dates:', len(excluded))
display(missing[missing.gt(0)].rename('Missing observations').to_frame())

Excluded Cboe-only historical dates: 27


,Missing observations
MSFT_Return,1
GOOGL_Return,1
NVDA_Return,1
AAPL_Return,1
AMZN_Return,1
META_Return,1
GSPC_Return,1


The seven missing values are the six stock returns and the S&P 500 return on 2018-01-02. No preceding close was downloaded, so those returns are left undefined. Every later return is calculated from the previous trading day's Close. Calendar fields are stored as integers; later analyses choose any additional encoding they need.

## Large historical moves

Returns are in percentage points: a value of 1.5 means 1.5%. The 15% threshold only flags observations for inspection. It never clips or replaces prices or returns. The saved CSV records a source URL for each historical flag; the table below shows the event notes. Earnings announcements and market reports provide the context. For the two March 2020 observations, the Federal Reserve announcements establish market context rather than prove a single cause.

In [4]:
moves = pd.read_csv(ROOT / 'outputs/metrics/phase1_large_moves.csv')
assert len(moves) == 14
assert moves['Source'].notna().all()
display(moves[['Date', 'Symbol', 'Return', 'Note']])

,Date,Symbol,Return,Note
0,2018-07-26,META,-18.960922,First session after Facebook's Q2 2018 results...
1,2018-11-16,NVDA,-18.755865,First session after NVIDIA's Q3 FY2019 results...
2,2020-03-16,NVDA,-18.452086,COVID-19 market disruption after the Federal R...
3,2020-03-24,NVDA,17.156426,COVID-19 market rebound following the Federal ...
4,2022-02-03,META,-26.390095,First session after Meta's Q4 2021 results and...
5,2022-04-28,META,17.593598,First session after Meta's Q1 2022 results; re...
6,2022-10-27,META,-24.557081,First session after Meta's Q3 2022 results and...
7,2023-02-02,META,23.282400,First session after Meta's Q4 2022 results and...
8,2023-05-25,NVDA,24.369636,First session after NVIDIA's Q1 FY2024 results...
9,2024-02-02,META,20.317643,"First session after Meta's Q4 2023 results, fi..."


## Separate unseen master

The same function builds January-July 2026 with the same schema. The first return uses the final historical close, from 2025-12-31. This is past information at that date. The historical build never depends on unseen observations.

Only data integrity, coverage, and the boundary-return calculation are checked here. Unseen large moves are saved as flags without event investigation. Unseen data are not used for EDA, pair selection, fitting, or tuning.

In [5]:
complete_run = build_master(CONFIG)
assert sha256(ROOT / 'data/processed/master_historical.csv') == historical_hash
unseen = pd.read_csv(ROOT / 'data/processed/master_unseen.csv', index_col='Date', parse_dates=True, float_precision='round_trip')
assert unseen.shape == (145, 48)
assert list(unseen.columns) == list(historical.columns)
assert unseen.notna().all().all()
excluded = pd.read_csv(ROOT / 'data/interim/excluded_rows.csv')
assert excluded.groupby('Period').size().to_dict() == {'historical': 27, 'unseen': 5}
display(pd.DataFrame(complete_run['datasets']).T)
display(excluded.groupby('Period').size().rename('Excluded Cboe-only dates').to_frame())

,rows,columns_including_date,first_date,last_date,missing_values
historical,2011,49,2018-01-02,2025-12-31,7
unseen,145,49,2026-01-02,2026-07-31,0


,Excluded Cboe-only dates
Period,
historical,27
unseen,5


## Independent return calculations

These three checks read Close directly from the raw CSV strings and calculate `100 * (current - previous) / previous` with Decimal arithmetic. The build calculates `100 * (current / previous - 1)` with pandas. Agreement is required within 0.0000000001 percentage points. The AAPL sample also checks a split date; the S&P 500 sample checks the year boundary.

In [6]:
return_checks = pd.read_csv(ROOT / 'outputs/metrics/phase1_return_checks.csv')
assert len(return_checks) == 3
assert return_checks['AbsoluteError'].le(1e-10).all()
display(return_checks)

,Period,Symbol,Date,PreviousDate,PreviousClose,Close,ExpectedReturn,ActualReturn,AbsoluteError
0,historical,MSFT,2018-01-03,2018-01-02,85.949997,86.349998,0.465389,0.465389,2.164935e-15
1,historical,AAPL,2020-08-31,2020-08-28,124.807503,129.039993,3.391215,3.391215,1.776357e-14
2,unseen,GSPC,2026-01-02,2025-12-31,6845.500000,6858.470215,0.189471,0.189471,7.549517e-15


## Repeatable outputs

The manifest records input hashes, settings, Python and pandas versions, build-file hashes, and output hashes. Generated CSVs use LF line endings on every operating system. The manifest omits a wall-clock build time so repeating the same build can reproduce it too.

In [7]:
manifest_path = ROOT / 'data/processed/master_manifest.json'
manifest_hash = sha256(manifest_path)
repeat_run = build_master(CONFIG)
assert complete_run == repeat_run
assert sha256(manifest_path) == manifest_hash
for relative, expected_hash in repeat_run['outputs'].items():
    path = ROOT / relative
    assert sha256(path) == expected_hash
    assert b'\r\n' not in path.read_bytes()
quality = pd.read_csv(ROOT / 'outputs/metrics/phase1_quality_summary.csv')
assert quality['Status'].eq('pass').all()
print(f"Identical hashes for {len(repeat_run['outputs'])} generated CSVs and the manifest.")
print(f"Quality checks passed: {len(quality)}")
display(quality.groupby(['Period', 'Status']).size().rename('Checks').to_frame())

Identical hashes for 8 generated CSVs and the manifest.
Quality checks passed: 64


,,Checks
Period,Status,
historical,pass,32
unseen,pass,32


The saved masters have 49 columns including Date. The historical file covers 2018-01-02 to 2025-12-31; the unseen file covers 2026-01-02 to 2026-07-31. All 32 Cboe-only dates are recorded in the exclusion log. The first historical returns are the only missing values.

The 14 historical large moves are retained. META's 2022-02-03 return is -26.3901%, and its 2023-02-02 return is +23.2824%. These are observations in the selected Yahoo snapshot, not corrections from another provider. Yahoo's known S&P 500 close discrepancy, opening-price differences, suspect highs/lows, and omitted dividends remain limitations described in the source analysis.

Phase 2 can use `data/processed/master_historical.csv`. Column meanings and allowed missing values are in [the data dictionary](../docs/data_dictionary.md). The unseen master remains separate for the evaluation phases.